# Sandbox: Nicolas

Conflict, Democracy and Arms project. This is your personal notebook for trying things out.
Only Nicolas edits this file, so it never causes a merge conflict with Yair.

**How every task works**

1. **What you need:** each new command is explained in one or two sentences, followed by a small example on made-up data. Run the example and look at the output. Change a number in it and run it again to see what happens.
2. **Your turn:** the same idea on our project data. Every step has its own empty cell. You write the code, using the example as a model.
3. **Hints:** only if you are stuck.
4. **Checkpoint:** a test cell. OK means the task is done.

When a task is done, paste the working code into the main notebook listed below, then commit and push.

| Part | Task | Paste into |
| --- | --- | --- |
| Practice | Merging on toy data | nothing, just for learning |
| A | V-Dem: democracy and regime per country-year | `03_vdem.ipynb` |
| B | UCDP: conflict per country-year and conflict periods | `02_ucdp.ipynb` |
| C | Descriptives: conflict by regime type | `06_analysis.ipynb`, Part A |
| D | Chart functions: regime bands, conflict bars | `07_charts.ipynb`, then `app/panels_conflict.py` |

Variable, column and file names come from `CONVENTIONS.md`. Use them exactly.

## Command card

Everything you need in this notebook, with the lecture where you saw it. Come back here when you forget a command.

| Command | What it does | Lecture |
| --- | --- | --- |
| `pd.read_csv(path)`, `pd.read_excel(path)` | load a file into a table (DataFrame) | week 3, week 5 joins |
| `df.head()`, `df.shape`, `df.columns`, `df.info()` | look at a table | week 3 |
| `df["col"]` | one column | week 3 |
| `df[df["year"] >= 1950]` | keep rows where a condition is True | week 3 cleaning |
| `cond1 & cond2`, `~cond` | both conditions, "not" | week 3 cleaning |
| `df["col"].between(a, b)`, `df["col"].isin([...])` | conditions for ranges and lists | week 3, week 4 |
| `df.rename(columns={"old": "new"})` | rename columns | week 3 |
| `df[["a", "b"]]` | keep only some columns | week 3 |
| `df.dropna(subset=["col"])`, `df["col"].isna()` | missing values | week 3 cleaning |
| `df["col"].astype(int)` | change the type | week 3 cleaning |
| `df.drop_duplicates()`, `df.duplicated()` | duplicates | week 3, week 5 joins |
| `df.sort_values(["a", "b"])` | sort | week 3 |
| `df.head(n)`, `df["col"].unique()`, `df["col"].value_counts()` | first rows, distinct values, counts per value | week 3, week 4 |
| `df["col"].fillna(0)` | replace missing values | week 3 cleaning |
| `df["col"].map(dictionary)` | translate values with a dictionary | week 4 Disney |
| `df["col"].str.endswith("*")` | True where the text ends with `*` | new, works like `str.contains` |
| `df.drop(columns="col")` | remove a column | week 3 |
| `pd.concat([df1, df2])` | stack tables | week 5 joins |
| `text.split(",")`, `text.strip()` | split or clean one piece of text (plain Python) | week 2 |
| `df["col"].str.strip()`, `df["col"].str.contains("x")` | the same for a whole column | week 3 cleaning |
| `for x in my_list:` | loop | week 2 |
| `for a, b in zip(df["a"], df["b"]):` | loop over two columns at once | week 4 |
| `my_list.append(x)` | add to a list | week 2 |
| `d = {}`, `d[key] = value`, `d.get(key)` | dictionary | week 2 |
| `pd.DataFrame(list_of_dicts)` | turn a list of dictionaries into a table | week 3 |
| `df.groupby("a").agg(n=("b", "count"))` | one row per group with summaries; functions: `"count"`, `"nunique"` (distinct values), `"sum"`, `"mean"`, `"max"` | week 4 Disney |
| `df.groupby("a")["b"].transform("mean")` | group value repeated on every row | week 4 Disney |
| `for name, group in df.groupby("a"):` | loop over groups | week 5 data viz |
| `pd.merge(left, right, on=[...], how="left", validate=..., indicator=True)` | join two tables | week 5 joins |
| `df.pivot(index=..., columns=..., values=...)` | long to wide | week 4 |
| `df.to_csv(path, index=False)` | save a table | week 3 |
| `fig, ax = plt.subplots()`, `ax.plot(x, y)`, `ax.bar(...)`, `ax.barh(...)` | charts | week 5 data viz |

If you see a `SettingWithCopyWarning`, add `.copy()` after the filter that made the table, for example `df = panel[panel["iso3"] == "UKR"].copy()`.

## Setup

This cell loads the packages and sets the folder paths. Run it every time you open the notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

---
## Practice: merging (do this before any real merge)

We never merge big tables without testing on tiny ones first.

#### What you need: merging two tables

A merge puts two tables side by side. Rows are matched by the value in one or more key columns.

- `on=` names the key column(s); for two keys, write a list: `on=["iso3", "year"]`.
- `how="inner"` keeps only keys found in both tables; `how="left"` keeps every row of the left table and leaves gaps (NaN) where the right table has nothing.
- `validate="one_to_one"` or `"many_to_one"` makes pandas stop with an error if a key appears more often than it should. This protects you from rows multiplying by accident.
- `indicator=True` adds a column `_merge` that says for every row whether it was found in both tables (`both`) or only in the left one (`left_only`).

*Seen in the lecture:* week 5, page `lecture_week_5_joins.py` (sections on left joins, `validate` and `indicator`)

In [ ]:
# Example: run this cell and look at the output
students = pd.DataFrame({"name": ["Anna", "Ben", "Cara"], "age": [21, 23, 22]})
grades = pd.DataFrame({"name": ["Ben", "Cara", "Dan"], "grade": [5.5, 4.0, 6.0]})

both = pd.merge(students, grades, on="name", how="inner")
print("inner:")
print(both)

everyone = pd.merge(students, grades, on="name", how="left", validate="one_to_one", indicator=True)
print("left:")
print(everyone)
print(everyone["_merge"].value_counts())

#### Your turn

Run the cell below. It creates two small tables from our project world.

In [ ]:
demo = pd.DataFrame({
    "iso3": ["CHE", "IND", "UKR"],
    "year": [2000, 2000, 2000],
    "democracy_index": [0.90, 0.70, 0.45],
})
arms = pd.DataFrame({
    "iso3": ["IND", "UKR", "SAU"],
    "year": [2000, 2000, 2000],
    "arms_imports_tiv": [900, 20, 1500],
})
print(demo)
print(arms)

**Step 1.** Predict first, in a comment: which countries appear in both tables? Which only in one?

In [ ]:
# your code here

**Step 2.** Inner merge of `demo` and `arms` on `["iso3", "year"]`. Call it `m_inner` and print it. Which country is missing, and why?

In [ ]:
# your code here

**Step 3.** Left merge with `demo` on the left, `validate="one_to_one"` and `indicator=True`. Call it `m_left` and print it. What is in `arms_imports_tiv` for CHE, and what does `_merge` say?

In [ ]:
# your code here

**Step 4.** In our project, no arms delivery means zero imports. Replace the missing value: `m_left["arms_imports_tiv"] = m_left["arms_imports_tiv"].fillna(0)`. Print again.

In [ ]:
# your code here

In [ ]:
arms_dup = pd.concat([arms, pd.DataFrame({"iso3": ["IND"], "year": [2000], "arms_imports_tiv": [5]})])
print(arms_dup)

**Step 5.** Run the cell above first, it makes `arms_dup` with India twice. Then try a left merge of `demo` with `arms_dup` with `validate="one_to_one"`. Read the error message. Then do it without `validate`, call it `m_dup`, and count the rows. What happened to India?

In [ ]:
# your code here

**Step 6.** Write two rules in your own words (as a comment): what does `how="left"` keep, and what does `validate` protect you from?

In [ ]:
# your rules here

**Checkpoint.**

In [ ]:
assert len(m_inner) == 2, "inner keeps only countries in both tables"
assert len(m_left) == 3, "left keeps every row of demo"
assert sorted(m_left["iso3"]) == ["CHE", "IND", "UKR"], "a left merge keeps the rows of the LEFT table (demo)"
assert "_merge" in m_left.columns, "use indicator=True"
assert m_left["arms_imports_tiv"].isna().sum() == 0, "fill the missing value with 0"
assert len(m_dup) == 4, "the duplicate India row in arms_dup creates an extra row"
print("OK")

---
## Part A: V-Dem (03)

Do this first. 01 Crosswalk and 05 Merge both need `vdem.csv`.

### A1 Load the V-Dem file

**Goal:** a table `vdem_raw` with only the six columns we need.

#### What you need: loading a file and keeping some columns

`pd.read_csv(path)` loads a CSV file. The V-Dem file has more than 4,000 columns, so we keep only the ones we need right after loading: `df[["a", "b"]]` with a list of column names. `.copy()` makes it a separate table.

*Seen in the lecture:* week 3, `01_intro_pandas_class.ipynb` and `02_data_cleaning_full.ipynb`

In [ ]:
# Example: run this cell and look at the output
cities = pd.DataFrame({"city": ["Zurich", "Bern"], "pop": [420000, 135000], "canton": ["ZH", "BE"], "lake": ["yes", "no"]})
cities.to_csv("example_cities.csv", index=False)      # make a small file to practise with

loaded = pd.read_csv("example_cities.csv")
small = loaded[["city", "pop"]].copy()
print(loaded.shape, small.shape)
print(small)

#### Your turn

**Step 1.** Load `RAW + "vdem/V-Dem-CY-Core-v16.csv"` into a table called `vdem_full`. It is large (over 4,000 columns), so this can take up to a minute, and pandas may show a `DtypeWarning` about mixed types. Both are harmless. Print its shape.

In [ ]:
# your code here

**Step 2.** Make a list `cols` with the six names `country_name`, `country_text_id`, `COWcode`, `year`, `v2x_polyarchy`, `v2x_regime`. Keep only those columns: `vdem_raw = vdem_full[cols].copy()`. Print the shape and the first rows.

In [ ]:
# your code here

**Step 3.** Count missing values per column with `vdem_raw.isna().sum()`. Write in a comment which columns have missing values.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "vdem_raw" in globals(), "create a table called vdem_raw"
assert list(vdem_raw.columns) == ["country_name", "country_text_id", "COWcode", "year", "v2x_polyarchy", "v2x_regime"]
assert vdem_raw["year"].min() < 1950
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A2 Filter years and rename

**Goal:** a table `vdem` with years 1950 to 2025 and the column names from CONVENTIONS.md.

#### What you need: filtering rows, renaming, dropping missing values, changing types

- A condition like `df["year"] >= 1950` gives True or False for every row. `df[condition]` keeps the True rows. Combine conditions with `&`.
- `df.rename(columns={"old": "new"})` renames. Without `columns=` nothing happens and there is no error.
- `df.dropna(subset=["col"])` removes rows where that column is empty.
- `df["col"].astype(int)` turns decimals into whole numbers. It only works when nothing is missing.

*Seen in the lecture:* week 3, `02_data_cleaning_full.ipynb`; week 4, `disney_cleaning_analysis.ipynb`

In [ ]:
# Example: run this cell and look at the output
shows = pd.DataFrame({"title": ["A", "B", "C", "D"], "yr": [1990, 2005, 2012, 2030], "score": [7.0, None, 8.0, 6.0]})

recent = shows["yr"] >= 2000
not_future = shows["yr"] <= 2025
shows2 = shows[recent & not_future].copy()
shows2 = shows2.rename(columns={"yr": "year"})
print(shows2)

shows3 = shows2.dropna(subset=["score"]).copy()
shows3["score"] = shows3["score"].astype(int)
print(shows3)

#### Your turn

**Step 1.** Make the conditions `from_1950` (year at least 1950) and `until_2025` (year at most 2025). Print how many rows are True for each with `.sum()`.

In [ ]:
# your code here

**Step 2.** Keep the rows where both are True and call the result `vdem` (add `.copy()`). Print the shape before and after.

In [ ]:
# your code here

**Step 3.** Rename with a dictionary `new_names`: `country_text_id` → `iso3`, `COWcode` → `cow_code`, `v2x_polyarchy` → `democracy_index`, `v2x_regime` → `regime_type`. Print `vdem.columns`.

In [ ]:
# your code here

**Step 4.** Look at the rows where `regime_type` is missing: `vdem[vdem["regime_type"].isna()]`. Decide whether to drop them. Dropping them also drops those country-years from the merge in 05, so write the decision and the number of rows in the decisions table. Then do it.

In [ ]:
# your code here

**Step 5.** Make `regime_type` a whole number with `astype(int)`. Print `vdem["regime_type"].unique()`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "vdem" in globals()
for col in ["iso3", "country_name", "cow_code", "year", "democracy_index", "regime_type"]:
    assert col in vdem.columns, f"missing column {col}"
assert vdem["year"].between(1950, 2025).all()
assert vdem["democracy_index"].dropna().between(0, 1).all()
assert vdem["regime_type"].dtype.kind == "i", "regime_type must be a whole number (astype(int))"
assert set(vdem["regime_type"].unique()) <= {0, 1, 2, 3}
assert vdem["iso3"].isna().sum() == 0
assert vdem.duplicated(["iso3", "year"]).sum() == 0, "one row per iso3-year"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A3 Regime labels

**Goal:** a column `regime_label` with readable names, spelled exactly as below (the charts and the app depend on it).

#### What you need: translating codes with a dictionary

A dictionary maps keys to values. `.map(dictionary)` translates every value in a column. You can also do it with a loop that fills a list.

*Seen in the lecture:* dictionaries: week 2; `.map`: week 4 Disney notebook

In [ ]:
# Example: run this cell and look at the output
grades = pd.DataFrame({"student": ["Anna", "Ben", "Cara"], "grade_code": [1, 3, 2]})
words = {1: "fail", 2: "pass", 3: "good"}

print(words[3])                                     # one lookup

labels = []                                         # with a loop
for code in grades["grade_code"]:
    labels.append(words[code])
grades["grade_word"] = labels

grades["grade_word_map"] = grades["grade_code"].map(words)   # the same with map
print(grades)

#### Your turn

**Step 1.** Write a dictionary `labels` with 0 = `closed autocracy`, 1 = `electoral autocracy`, 2 = `electoral democracy`, 3 = `liberal democracy` (all lower case).

In [ ]:
# your code here

**Step 2.** Make the column `regime_label`, with a loop or with `.map(labels)`.

In [ ]:
# your code here

**Step 3.** Count how often each label appears: `vdem["regime_label"].value_counts()`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert vdem["regime_label"].isna().sum() == 0
assert set(vdem["regime_label"]) == {"closed autocracy", "electoral autocracy", "electoral democracy", "liberal democracy"}, "check spelling and lower case"
assert (vdem[vdem["regime_type"] == 3]["regime_label"] == "liberal democracy").all()
assert (vdem[vdem["regime_type"] == 0]["regime_label"] == "closed autocracy").all()
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A4 Regime periods for the timeline

**Goal:** `vdem_regime_spans`: one row per uninterrupted regime period, with exactly the columns `iso3`, `regime_label`, `start`, `end`.

#### What you need: turning years into periods (a loop that remembers)

For the timeline we need periods ("1994 to 2004: electoral democracy") and not one row per year. The idea: go through the years in order, remember when the current period started, and write a period down when it ends.

The example below does this for one person's weather diary. A period ends when the weather changes. Read it line by line; every line is plain Python from weeks 2 and 4.

*Seen in the lecture:* loops, `zip` and dictionaries: week 2 and week 4 fundamentals

In [ ]:
# Example: run this cell and look at the output
diary = pd.DataFrame({
    "day": [1, 2, 3, 4, 5, 6],
    "weather": ["sun", "sun", "rain", "rain", "rain", "sun"],
})

periods = []                     # finished periods are collected here
start = None                     # first day of the current period
current = None                   # weather of the current period
prev_day = None                  # the day we looked at last

for day, weather in zip(diary["day"], diary["weather"]):
    if prev_day is None:         # the very first row: a period starts
        start = day
        current = weather
    elif weather != current:     # the weather changed: the old period ends
        periods.append({"weather": current, "start": start, "end": prev_day})
        start = day
        current = weather
    prev_day = day

periods.append({"weather": current, "start": start, "end": prev_day})   # the last period
print(pd.DataFrame(periods))

#### What you need: looping over groups

`for name, group in df.groupby("col"):` runs the loop once per group. `name` is the group's value (for example a country code), and `group` is a small table with only that group's rows.

*Seen in the lecture:* week 5, page `lecture_week_5_data_viz.py` (`for name, g in df.groupby("country"):`)

In [ ]:
# Example: run this cell and look at the output
sales = pd.DataFrame({
    "shop": ["A", "A", "B", "B", "B"],
    "day": [1, 2, 1, 2, 3],
    "amount": [10, 12, 5, 7, 9],
})
for shop, rows in sales.groupby("shop"):
    print("shop", shop, "has", len(rows), "days, total", rows["amount"].sum())

#### Your turn

**Step 1.** Sort `vdem` by `iso3` and `year`. Make `ukr = vdem[vdem["iso3"] == "UKR"]`, print it and note by eye where the regime changes.

In [ ]:
# your code here

**Step 2.** Copy the weather example and change it for **Ukraine only**: loop over `zip(ukr["year"], ukr["regime_label"])`. Two changes compared to the example: a period also ends when a year is missing (`year != prev_year + 1`), and each period gets the key `iso3`. The dictionary keys are `iso3`, `regime_label`, `start`, `end`. Print the periods and compare with step 1.

In [ ]:
# your code here

**Step 3.** Put your Ukraine code inside a loop over all countries: `for code, rows in vdem.groupby("iso3"):`. Inside, `periods` keeps collecting, but `start`, `current` and `prev_year` start fresh for each country. At the end: `vdem_regime_spans = pd.DataFrame(periods)`. Print the rows for `"CHE"`.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

In the example the period ends with `elif weather != current:`. You need two reasons to end a period, joined with `or`.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

The line `periods = []` comes before the country loop. The lines `prev_year = None` come inside it, before the year loop. The `periods.append(...)` for the last period comes after the year loop, still inside the country loop.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(vdem_regime_spans.columns) == ["iso3", "regime_label", "start", "end"]
assert (vdem_regime_spans["end"] >= vdem_regime_spans["start"]).all()
years_covered = (vdem_regime_spans["end"] - vdem_regime_spans["start"] + 1).sum()
assert years_covered == len(vdem), f"periods cover {years_covered} years, vdem has {len(vdem)} rows (forgot the last period, or bridged a gap?)"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### A5 Save

**Goal:** the two clean files in `data/clean/`.

#### Your turn

**Step 1.** Save `vdem` as `vdem.csv` and `vdem_regime_spans` as `vdem_regime_spans.csv`: `vdem.to_csv(CLEAN + "vdem.csv", index=False)`. `index=False` stops pandas from writing an extra column.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert os.path.exists(CLEAN + "vdem.csv") and os.path.exists(CLEAN + "vdem_regime_spans.csv")
assert "Unnamed: 0" not in pd.read_csv(CLEAN + "vdem.csv").columns, "save with index=False"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

---
## Part B: UCDP (02)

Needs `crosswalk_gw.csv` from 01 Crosswalk.

### B1 Load and inspect

**Goal:** `ucdp_raw` loaded, understood, and limited to 1950 onwards.

#### What you need: loading Excel and searching text

`pd.read_excel(path)` works like `read_csv`. `df["col"].astype(str).str.contains(",")` is True for every row whose text contains a comma.

*Seen in the lecture:* week 3, `02_data_cleaning_full.ipynb` (`str.contains`)

In [ ]:
# Example: run this cell and look at the output
teams = pd.DataFrame({"match": [1, 2, 3], "teams": ["CH", "CH, DE", "FR, IT"]})
has_comma = teams["teams"].astype(str).str.contains(",")
print(has_comma)
print(has_comma.sum(), "matches with two teams")
print(teams[has_comma])

#### Your turn

**Step 1.** Load `RAW + "UcdpPrioConflict_v26_1.xlsx"` with `pd.read_excel` into `ucdp_raw`. Print the shape and the column names.

In [ ]:
# your code here

**Step 2.** Print the columns `conflict_id`, `year`, `gwno_loc`, `intensity_level` for the first 10 rows. Write in a comment what one row stands for.

In [ ]:
# your code here

**Step 3.** Find the rows where `gwno_loc` contains a comma, like in the example. How many are there? Print five. Why can one conflict-year have several countries?

In [ ]:
# your code here

**Step 4.** Keep only rows with `year >= 1950` and overwrite `ucdp_raw` (add `.copy()`). Our panel starts in 1950, and 01 Crosswalk only matches codes from 1950 on. Print how many rows were removed.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

One row is one conflict in one year. Wars between states involve several countries, so UCDP lists all their codes in one cell, for example "651, 666".

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "ucdp_raw" in globals()
for col in ["conflict_id", "year", "gwno_loc", "intensity_level"]:
    assert col in ucdp_raw.columns
assert ucdp_raw["year"].min() >= 1950, "keep only years from 1950 on (step 4)"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B2 One row per country per conflict-year

**Goal:** `ucdp_long`: a row with "651, 666" becomes two rows, one with `gw_code` 651 and one with 666.

#### What you need: splitting text and building a new table row by row

`"651, 666".split(",")` gives the list `["651", " 666"]`. `.strip()` removes the spaces and `int()` makes a number. To build a new table, collect one dictionary per row in a list and call `pd.DataFrame(list)` at the end.

*Seen in the lecture:* `split` and `strip`: week 2; `zip` and lists of dictionaries: week 4

In [ ]:
# Example: run this cell and look at the output
matches = pd.DataFrame({"match": [1, 2], "teams": ["CH", "CH, DE"], "goals": [2, 3]})

rows = []
for match, teams, goals in zip(matches["match"], matches["teams"], matches["goals"]):
    for team in str(teams).split(","):
        rows.append({"match": match, "team": team.strip(), "goals": goals})

matches_long = pd.DataFrame(rows)
print(matches_long)

#### Your turn

**Step 1.** Try the idea on one value first: split the text `"651, 666"`, loop over the pieces, and print each one as a number.

In [ ]:
# your code here

**Step 2.** Now the whole table, like the example: loop over `zip(ucdp_raw["conflict_id"], ucdp_raw["year"], ucdp_raw["gwno_loc"], ucdp_raw["intensity_level"])`. For every code, append a dictionary with the keys `conflict_id`, `year`, `gw_code` (as `int`), `intensity_level`.

In [ ]:
# your code here

**Step 3.** Make the table: `ucdp_long = pd.DataFrame(rows)`. Print the number of rows of `ucdp_raw` and of `ucdp_long`. By how many did it grow? Compare with `ucdp_raw["gwno_loc"].astype(str).str.count(",").sum()`, the total number of commas (a cell with three codes has two commas).

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_long.columns) == ["conflict_id", "year", "gw_code", "intensity_level"]
assert ucdp_long["gw_code"].dtype.kind == "i", "gw_code must be a whole number (int)"
expected = len(ucdp_raw) + ucdp_raw["gwno_loc"].astype(str).str.count(",").sum()
assert len(ucdp_long) == expected, f"expected {expected} rows: one per code"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B3 Add iso3 with a merge

**Goal:** every row in `ucdp_long` gets an `iso3`, and no row is lost or added.

#### What you need: merging two tables

A merge puts two tables side by side. Rows are matched by the value in one or more key columns.

- `on=` names the key column(s); for two keys, write a list: `on=["iso3", "year"]`.
- `how="inner"` keeps only keys found in both tables; `how="left"` keeps every row of the left table and leaves gaps (NaN) where the right table has nothing.
- `validate="one_to_one"` or `"many_to_one"` makes pandas stop with an error if a key appears more often than it should. This protects you from rows multiplying by accident.
- `indicator=True` adds a column `_merge` that says for every row whether it was found in both tables (`both`) or only in the left one (`left_only`).

*Seen in the lecture:* week 5, page `lecture_week_5_joins.py` (sections on left joins, `validate` and `indicator`)

In [ ]:
# Example: run this cell and look at the output
students = pd.DataFrame({"name": ["Anna", "Ben", "Cara"], "age": [21, 23, 22]})
grades = pd.DataFrame({"name": ["Ben", "Cara", "Dan"], "grade": [5.5, 4.0, 6.0]})

both = pd.merge(students, grades, on="name", how="inner")
print("inner:")
print(both)

everyone = pd.merge(students, grades, on="name", how="left", validate="one_to_one", indicator=True)
print("left:")
print(everyone)
print(everyone["_merge"].value_counts())

#### Your turn

**Step 1.** Load `CLEAN + "crosswalk_gw.csv"` into `crosswalk_gw`. Print the first rows. Which column also exists in `ucdp_long`?

In [ ]:
# your code here

**Step 2.** Save the number of rows: `n_before = len(ucdp_long)`.

In [ ]:
# your code here

**Step 3.** Left merge `crosswalk_gw` onto `ucdp_long` on `gw_code`, with `validate="many_to_one"` (many conflict rows can share one code, but every code appears once in the crosswalk) and `indicator=True`. Overwrite `ucdp_long`. Print the length; it must equal `n_before`.

In [ ]:
# your code here

**Step 4.** Count `ucdp_long["_merge"].value_counts()`. Rows with `left_only` found no iso3: print their unique `gw_code` values. Fix those in 01 Crosswalk, not here. Then remove the `_merge` column: `ucdp_long = ucdp_long.drop(columns="_merge")`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

If the merge stops with a `MergeError`, a `gw_code` appears twice in the crosswalk. That is the `validate` check doing its job: fix the crosswalk.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert len(ucdp_long) == n_before, "the merge added rows"
assert "_merge" not in ucdp_long.columns, "drop the _merge column after checking it"
assert ucdp_long["iso3"].isna().sum() == 0, "some codes have no iso3: fix crosswalk_gw in 01"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B4 Country-year table

**Goal:** `ucdp_cy` with one row per `iso3` and `year`, and the columns `iso3`, `year`, `n_conflicts`, `max_intensity`, `conflict_active`.

#### What you need: summaries per group with groupby and agg

`df.groupby(["a", "b"], as_index=False).agg(new_name=("column", "function"))` makes one row per combination of `a` and `b`. Each `new_name=(...)` is one summary: `"nunique"` counts different values, `"max"` takes the largest, `"sum"` adds up, `"mean"` averages.

*Seen in the lecture:* week 4, `disney_cleaning_analysis.ipynb` (`groupby(...).agg(movie_count=(...))`)

In [ ]:
# Example: run this cell and look at the output
orders = pd.DataFrame({
    "shop": ["A", "A", "A", "B"],
    "day": [1, 1, 2, 1],
    "order_id": [101, 102, 103, 104],
    "size": [1, 3, 2, 2],
})
per_day = orders.groupby(["shop", "day"], as_index=False).agg(
    n_orders=("order_id", "nunique"),
    biggest=("size", "max"),
)
print(per_day)

#### Your turn

**Step 1.** Do one country-year by hand first: filter `ucdp_long` to UKR 2015. Count the different `conflict_id` values and take the highest `intensity_level`. Write down both numbers.

In [ ]:
# your code here

**Step 2.** Now all at once, like the example: group by `["iso3", "year"]` with `as_index=False` and `agg` with `n_conflicts=("conflict_id", "nunique")` and `max_intensity=("intensity_level", "max")`. Call it `ucdp_cy`. Check that UKR 2015 shows your hand numbers.

In [ ]:
# your code here

**Step 3.** Add the column `conflict_active` with the value 1 for every row: `ucdp_cy["conflict_active"] = 1`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_cy.columns) == ["iso3", "year", "n_conflicts", "max_intensity", "conflict_active"]
assert ucdp_cy.duplicated(["iso3", "year"]).sum() == 0
assert set(ucdp_cy["max_intensity"].unique()) <= {1, 2}
assert (ucdp_cy["conflict_active"] == 1).all()
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### B5 Conflict periods and save

**Goal:** `ucdp_spans` (conflict periods for the timeline bars) with exactly the columns `iso3`, `start`, `end`, `max_intensity`, and both CSV files saved.

#### What you need: keeping the highest value during a period

Same loop idea as in A4. This time a period only ends when there is a gap in the years, and we also remember the highest value seen during the period with `max(old, new)`.

*Seen in the lecture:* loops and `max`: week 2

In [ ]:
# Example: run this cell and look at the output
print(max(1, 2), max(2, 1))       # max of two numbers

readings = pd.DataFrame({"day": [1, 2, 3, 6, 7], "level": [1, 2, 1, 1, 1]})
periods = []
prev_day = None
for day, level in zip(readings["day"], readings["level"]):
    if prev_day is None:
        start = day
        highest = level
    elif day != prev_day + 1:          # a gap: the old period ended
        periods.append({"start": start, "end": prev_day, "highest": highest})
        start = day
        highest = level
    else:                              # the period continues
        highest = max(highest, level)
    prev_day = day
periods.append({"start": start, "end": prev_day, "highest": highest})
print(pd.DataFrame(periods))

#### Your turn

**Step 1.** Sort `ucdp_cy` by `iso3` and `year`.

In [ ]:
# your code here

**Step 2.** Adapt the example for **one country first** (`"COL"`), using `year` and `max_intensity`, and the keys `iso3`, `start`, `end`, `max_intensity`.

In [ ]:
# your code here

**Step 3.** Wrap it in `for code, rows in ucdp_cy.groupby("iso3"):` as in A4, make `ucdp_spans = pd.DataFrame(periods)`, and save `ucdp_cy.csv` and `ucdp_spans.csv` with `index=False`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_spans.columns) == ["iso3", "start", "end", "max_intensity"]
assert (ucdp_spans["end"] >= ucdp_spans["start"]).all()
assert (ucdp_spans["end"] - ucdp_spans["start"] + 1).sum() == len(ucdp_cy), "spans must cover every conflict year once"
keys = set(zip(ucdp_cy["iso3"], ucdp_cy["year"]))
n_starts = sum(1 for (cc, y) in keys if (cc, y - 1) not in keys)
assert len(ucdp_spans) == n_starts, "one span per uninterrupted run of conflict years"
assert set(ucdp_spans["max_intensity"]) <= {1, 2}
war_years = set(zip(ucdp_cy[ucdp_cy["max_intensity"] == 2]["iso3"], ucdp_cy[ucdp_cy["max_intensity"] == 2]["year"]))
runs_with_war = 0
for code, start, end in zip(ucdp_spans["iso3"], ucdp_spans["start"], ucdp_spans["end"]):
    if any((code, y) in war_years for y in range(start, end + 1)):
        runs_with_war += 1
assert (ucdp_spans["max_intensity"] == 2).sum() == runs_with_war, "keep the highest intensity of each period (the else branch with max)"
assert os.path.exists(CLEAN + "ucdp_cy.csv") and os.path.exists(CLEAN + "ucdp_spans.csv")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

---
## Part C: Descriptives (06 Part A)

Needs `panel.csv` from 05 Merge.

### C1 Conflict by regime type

**Goal:** a dictionary `conflict_share` with the share of country-years in conflict for each regime label.

#### What you need: the mean of a 0/1 column is a share

If a column contains only 0 and 1, its mean is the share of 1s. Grouped, it gives the share per group.

*Seen in the lecture:* week 4, `groupby` and `mean`

In [ ]:
# Example: run this cell and look at the output
visits = pd.DataFrame({"weekday": ["Mon", "Mon", "Tue", "Tue", "Tue"], "rained": [1, 0, 1, 1, 0]})
mon = visits[visits["weekday"] == "Mon"]
print("by hand:", mon["rained"].sum() / len(mon))
print(visits.groupby("weekday")["rained"].mean())

#### Your turn

**Step 1.** Load `CLEAN + "panel.csv"` into `panel`.

In [ ]:
# your code here

**Step 2.** By hand for one label: filter `regime_label == "closed autocracy"`, count the rows, count the rows with `conflict_active == 1`, divide. Print the share.

In [ ]:
# your code here

**Step 3.** For all four labels: an empty dictionary `conflict_share = {}`, a loop over the labels, and `conflict_share[label] = share`. Print it. Then check with `panel.groupby("regime_label")["conflict_active"].mean()`.

In [ ]:
# your code here

**Step 4.** Write one sentence with the result as a comment.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert len(conflict_share) == 4
for label in conflict_share:
    expected = panel[panel["regime_label"] == label]["conflict_active"].mean()
    assert abs(conflict_share[label] - expected) < 1e-9, f"share for {label} is not right"
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

---
## Part D: Chart functions (07)

Each function draws on an `ax` it receives. Test it here, then copy it unchanged into `07_charts.ipynb` and from there into `app/panels_conflict.py`.

**Rules for every timeline** (see `CONVENTIONS.md`): `ax.set_xlim(1950, 2025)`, a short title with `ax.set_title`, `ax.set_yticks([])` when the y-axis has no numbers, and a dashed line `ax.axvline(year, linestyle="--", color="grey")` when `year` is given. Inside the function, use only the arguments (never a global table). The colour dictionaries at the top of the file are the one exception.

### D0 Load the files and agree on colours

**Goal:** the chart data loaded from the CSV files (as the app will do it), and one colour dictionary.

#### What you need: drawing charts on an ax

`fig, ax = plt.subplots()` makes a figure (`fig`) with one drawing area (`ax`). You draw on `ax` with `ax.plot` (lines), `ax.bar` (vertical bars) or `ax.barh` (horizontal bars).

Our chart functions get the `ax` as their first argument and only draw on it. Then the app can put five charts under each other with `plt.subplots(5, 1, sharex=True)`, and they share the same year axis automatically.

Useful: `ax.set_xlim(1950, 2025)` fixes the year range, `ax.axvline(2014, linestyle="--")` draws a dashed vertical line, `ax.set_title("...")`, `ax.set_ylabel("...")`, `ax.text(0.5, 0.5, "message", transform=ax.transAxes, ha="center")` writes a message in the middle.

*Seen in the lecture:* week 5, page `lecture_week_5_data_viz.py` (`fig, ax = plt.subplots()`, `ax.plot`, `plt.tight_layout()`)

In [ ]:
# Example: run this cell and look at the output
def temperature_line(ax, df, city, year=None):
    rows = df[df["city"] == city]
    ax.plot(rows["year"], rows["temp"])
    ax.set_xlim(2000, 2005)
    ax.set_title("Temperature, " + city)
    ax.set_ylabel("°C")
    if year is not None:
        ax.axvline(year, linestyle="--", color="grey")

weather = pd.DataFrame({
    "city": ["Zurich"] * 5 + ["Bern"] * 5,
    "year": [2000, 2001, 2002, 2003, 2004] * 2,
    "temp": [9.1, 9.4, 9.0, 10.2, 9.8, 8.7, 8.9, 8.6, 9.9, 9.3],
})

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4))
temperature_line(axes[0], weather, "Zurich", year=2003)
temperature_line(axes[1], weather, "Bern", year=2003)
plt.tight_layout()
plt.show()

#### Your turn

**Step 1.** Load `vdem_regime_spans.csv` and `ucdp_spans.csv` from `CLEAN` (`panel` is already loaded from Part C).

In [ ]:
# your code here

**Step 2.** Write a dictionary `regime_colours` with one colour (hex code, for example `"#1D4E89"`) for each of the four regime labels, and `intensity_colours` with colours for `"minor"` and `"war"`. Agree on them with Yair; they go to the top of `07_charts.ipynb` and of all three panels files.

In [ ]:
# your code here

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### D1 regime_bands(ax, spans, iso3, year=None)

**Goal:** coloured bands, one per regime period, all on one line.

#### What you need: horizontal bars for periods

`ax.barh(y, width, left=..., color=...)` draws horizontal bars. `left` is where a bar starts and `width` is how long it is. A period from 1994 to 2004 has `left=1994` and `width=11`. With `y=0` for every bar, all bars sit on one line, like a timeline.

*Seen in the lecture:* matplotlib basics from week 5; `barh` is the horizontal version of `bar`

In [ ]:
# Example: run this cell and look at the output
holidays = pd.DataFrame({"kind": ["ski", "beach", "ski"], "start": [2001, 2003, 2006], "end": [2002, 2004, 2008]})
colours = {"ski": "#4C78A8", "beach": "#F2C14E"}

fig, ax = plt.subplots(figsize=(8, 1.5))
for kind, start, end in zip(holidays["kind"], holidays["start"], holidays["end"]):
    ax.barh(0, end - start + 1, left=start, color=colours[kind])
ax.set_xlim(2000, 2010)
ax.set_yticks([])               # no y labels needed for one line
plt.show()

#### Your turn

**Step 1.** Without a function first: `fig, ax = plt.subplots(figsize=(10, 1.2))`. Filter `vdem_regime_spans` to `"UKR"`. Loop over `zip(...)` of `regime_label`, `start`, `end` and draw one `ax.barh(0, end - start + 1, left=start, color=regime_colours[label])` per period. Show it with `plt.show()`.

In [ ]:
# your code here

**Step 2.** Add the timeline rules: `ax.set_xlim(1950, 2025)`, `ax.set_yticks([])`, `ax.set_title("Regime")`, and the dashed line when a year is given (`if year is not None:`).

In [ ]:
# your code here

**Step 3.** Now wrap it in `def regime_bands(ax, spans, iso3, year=None):`. Inside, filter `spans` (the argument). The function only draws; it does not create a figure and does not call `plt.show()`. Test it like the temperature example: `fig, ax = plt.subplots(figsize=(10, 1.2))`, `regime_bands(ax, vdem_regime_spans, "UKR", 2014)`, `plt.show()`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

For a legend: `from matplotlib.patches import Patch`, then `ax.legend(handles=[Patch(color=c, label=l) for l, c in regime_colours.items()], ncol=4, fontsize=7, loc="upper left", bbox_to_anchor=(0, 1.6))`. Optional.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
regime_bands(ax, vdem_regime_spans, "CHE")
n_periods = len(vdem_regime_spans[vdem_regime_spans["iso3"] == "CHE"])
assert len(ax.patches) == n_periods, "one bar per period of the chosen country"
assert ax.get_xlim() == (1950, 2025), "use ax.set_xlim(1950, 2025)"
assert len(ax.lines) == 0, "no dashed line when year is None"
fig, ax = plt.subplots()
only_che = vdem_regime_spans[vdem_regime_spans["iso3"] == "CHE"]
regime_bands(ax, only_che, "UKR", year=2014)
assert len(ax.patches) == 0, "filter the argument spans, not the global table"
assert len(ax.lines) == 1, "one dashed line when a year is given"
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

### D2 conflict_bars(ax, spans, iso3, year=None)

**Goal:** bars for each conflict period, coloured by intensity, same look as D1.

#### What you need: a message when there is nothing to draw

Some countries had no conflict. The chart should then show a short message and no error. `ax.text(0.5, 0.5, "...", transform=ax.transAxes, ha="center", va="center")` writes text in the middle of the drawing area (0.5 means half way).

*Seen in the lecture:* matplotlib basics, week 5

In [ ]:
# Example: run this cell and look at the output
fig, ax = plt.subplots(figsize=(8, 1.2))
rows = pd.DataFrame({"start": [], "end": []})
if len(rows) == 0:
    ax.text(0.5, 0.5, "Nothing recorded", transform=ax.transAxes, ha="center", va="center", color="grey")
ax.set_xlim(2000, 2010)
ax.set_yticks([])
plt.show()

#### Your turn

**Step 1.** Same structure as D1 with `ucdp_spans`. The colour comes from `max_intensity`: translate 1 to `"minor"` and 2 to `"war"` with a small dictionary, then look up `intensity_colours`.

In [ ]:
# your code here

**Step 2.** Add the message for a country without conflict (`if len(rows) == 0:`), then the timeline rules, the title `"Conflict"` and the dashed year line.

In [ ]:
# your code here

**Step 3.** Wrap it in the function and test it with `"COL"` and with `"CHE"`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
fig, ax = plt.subplots()
conflict_bars(ax, ucdp_spans, "XXX")
assert len(ax.texts) == 1, "a country without conflict shows one message"
assert ax.get_xlim() == (1950, 2025)
fig, ax = plt.subplots()
only_col = ucdp_spans[ucdp_spans["iso3"] == "COL"]
conflict_bars(ax, only_col, "UKR")
assert len(ax.patches) == 0 and len(ax.texts) == 1, "filter the argument spans, not the global table"
fig, ax = plt.subplots()
conflict_bars(ax, ucdp_spans, "COL", year=1990)
assert len(ax.patches) == len(ucdp_spans[ucdp_spans["iso3"] == "COL"])
assert len(ax.lines) == 1
plt.close("all")
print("OK")

**In your own words** (one or two sentences): what did this task do, and why do we need it?

## Before you move code into the main notebook

- [ ] Kernel → Restart and Run All runs this sandbox from top to bottom without an error
- [ ] All checkpoints above print OK
- [ ] Names follow `CONVENTIONS.md`
- [ ] You wrote the "in your own words" sentence for every task
- [ ] You can explain every line to your partner without looking at the examples
- [ ] Copy only your working cells into the numbered main notebook (no examples, no experiments)
- [ ] Run the main notebook with Restart and Run All, then commit and push